# Student Placement Prediction - Step 6: Model Validation and Improvement

This notebook performs Step 6: Model Validation and Improvement:
1. **Feature Relationship Analysis**: Point-biserial correlations, Logistic Regression coefficients, and tree-based feature importances.
2. **5-Fold Stratified Cross-Validation**: Hyperparameter tuning on `X_train_processed` ONLY, optimizing for F1-score.
3. **Untouched Test Set Evaluation**: Single evaluation of best tuned models on `X_test_processed` (20,000 samples).
4. **Step 5 vs Step 6 Comparison**: Comparison against baseline model metrics.
5. **Predictive Performance Investigation**: Analysis of why accuracy remains ~55-57%.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from pathlib import Path
from train_test_split import execute_step_4
from model_validation import (
    analyze_feature_relationships,
    extract_feature_importances_and_coefs,
    perform_hyperparameter_tuning,
    evaluate_tuned_models_on_test
)

data_path = Path('../data/student_placement_prediction_dataset_2026-selected-columns.csv')
step4_data = execute_step_4(str(data_path))
X_train = step4_data['X_train_processed']
X_test = step4_data['X_test_processed']
y_train = step4_data['y_train']
y_test = step4_data['y_test']

## 1. Feature Relationship and Correlation Analysis

In [ ]:
corr_df = analyze_feature_relationships(X_train, y_train)
corr_df

## 2. Feature Importances and Coefficients

In [ ]:
imp_dict = extract_feature_importances_and_coefs(X_train, y_train)
print("=== Logistic Regression Coefficients ===")
display(imp_dict['logistic_regression_coefs'])

print("=== Random Forest Feature Importances ===")
display(imp_dict['random_forest_importances'])

print("=== Gradient Boosting Feature Importances ===")
display(imp_dict['gradient_boosting_importances'])

## 3. 5-Fold Stratified CV Hyperparameter Tuning (Training Data Only)

In [ ]:
tuning_results = perform_hyperparameter_tuning(X_train, y_train)
for model_name, res in tuning_results.items():
    print(f"Model: {model_name}")
    print(f"  Best 5-Fold CV F1-Score: {res['best_cv_f1']:.4f}")
    print(f"  Best Hyperparameters:   {res['best_params']}\n")

## 4. Final Evaluation of Tuned Models on Untouched Test Set

In [ ]:
eval_df, cms = evaluate_tuned_models_on_test(tuning_results, X_test, y_test)
eval_df

## 5. Comparison: Step 5 Baseline vs Step 6 Tuned Models

In [ ]:
step5_baseline = pd.DataFrame([
    {'Model': 'Gradient Boosting', 'Step5_F1': 0.6670, 'Step5_Acc': 0.5667},
    {'Model': 'Logistic Regression', 'Step5_F1': 0.6626, 'Step5_Acc': 0.5682},
    {'Model': 'Random Forest', 'Step5_F1': 0.6286, 'Step5_Acc': 0.5500}
])

comp_table = pd.merge(eval_df, step5_baseline, on='Model')
comp_table['F1_Improvement'] = comp_table['Test F1-score'] - comp_table['Step5_F1']
comp_table['Acc_Improvement'] = comp_table['Test Accuracy'] - comp_table['Step5_Acc']
comp_table[['Model', 'Step5_F1', 'Test F1-score', 'F1_Improvement', 'Step5_Acc', 'Test Accuracy', 'Acc_Improvement']]